# Solution B2 · Imputers compared: mean, median, KNN and iterative, with known truth

**Type**: Implement with scikit-learn

## Objective

Insert random missing values into complete data, impute them with four methods and measure the error with respect to the true
values **only on the removed cells**, to establish which method reconstructs them best and why.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.experimental import enable_iterative_imputer          # noqa: F401
from sklearn.impute import IterativeImputer, KNNImputer, SimpleImputer

rng = np.random.default_rng(0)
n = 800
X_true = rng.normal(size=(n, 2)) @ rng.normal(size=(2, 5)) + 0.5 * rng.normal(size=(n, 5))
X_true = (X_true - X_true.mean(axis=0)) / X_true.std(axis=0)                     # 5 standardized, correlated features
mask = rng.random(X_true.shape) < 0.20                                           # 20% of the cells missing, at random
X_miss = X_true.copy()
X_miss[mask] = np.nan
print("missing cells:", mask.mean().round(3), "| mean correlation between the features:",
      np.abs(np.corrcoef(X_true.T)[np.triu_indices(5, 1)]).mean().round(2))

## Commented solution

In [ ]:
imputers = {
    "mean": SimpleImputer(strategy="mean"),
    "median": SimpleImputer(strategy="median"),
    "knn": KNNImputer(n_neighbors=5),
    "iterative": IterativeImputer(random_state=0, max_iter=10),
}
rmse = {}
for name, imp in imputers.items():
    X_imp = imp.fit_transform(X_miss)
    rmse[name] = np.sqrt(np.mean((X_imp[mask] - X_true[mask]) ** 2))         # error ONLY on the missing cells
best = min(rmse, key=rmse.get)
print({k: round(v, 3) for k, v in rmse.items()}, "-> best:", best)

In [ ]:
shape_with_indicator = SimpleImputer(strategy="mean", add_indicator=True).fit_transform(X_miss).shape
relative_rmse = {k: v / rmse["mean"] for k, v in rmse.items()}
print("shape with the indicator:", shape_with_indicator)
print({k: round(v, 2) for k, v in relative_rmse.items()})

## Check

In [ ]:
with verify():
    check(set(imputers) == {"mean", "median", "knn", "iterative"}, "four imputers", "you need the keys mean, median, knn, iterative")
    check(set(rmse) == set(imputers) and all(np.isfinite(v) for v in rmse.values()), "one RMSE per method", "rmse must have a key for every imputer")
    check(0.9 < rmse["mean"] < 1.1, "the RMSE of the mean is about 1: the standard deviation of a standardized feature",
          "imputing with the mean the error is the standard deviation of the feature, that is about 1")
    check(rmse["knn"] < rmse["mean"] and rmse["iterative"] < rmse["mean"], "exploiting the other columns beats the mean",
          "knn and iterative should have a lower RMSE than the mean")
    check(best == min(rmse, key=rmse.get), "best consistent with rmse", "best = min(rmse, key=rmse.get)")
    check(best == "iterative", "with linear relationships between the columns the iterative imputer wins",
          "with data built from linear factors, IterativeImputer should win")
    check(shape_with_indicator == (800, 10), "add_indicator adds one column for every column with missing values (5 + 5)",
          "expected shape (800, 10)")
    check(abs(relative_rmse["mean"] - 1.0) < 1e-9 and relative_rmse["iterative"] < 0.8, "the iterative imputer reduces the error by more than 20%",
          "relative_rmse = rmse[name] / rmse['mean']")
done()

## What you learned

- With the mean the error is the standard deviation of the feature (about 1): the imputed value says nothing about the true value,
  so it is wrong on average by one standard deviation.
- KNN and Iterative use the other columns: when the features are correlated they reduce the error sharply. Where the
  columns are independent there is nothing to exploit and they go back to the level of the mean.
- `add_indicator=True` adds a binary column for every column that had missing values (here 5 + 5 = 10). With MCAR missing values
  it is not needed, with informative missing values (`income` in the loans) it can be the most useful signal.

## Common pitfall

Evaluating an imputer on the RMSE of all the cells (including the non-missing ones, which are unchanged) instead of only on the
removed ones: the result looks better than it is. And forgetting that the error is computed against a truth that in real
data you don't have: the simulation serves to choose the method, not to estimate the error in production.

## Interview question

> *Click on a question to show its answer.*

<details><summary>When would you use imputation with the mean instead of KNN or iterative?</summary>

When the columns are barely correlated (KNN and iterative have nothing to exploit), when the dataset is huge and the
computation cost matters, or when I need a robust method that is easy to explain and to put in production. With few missing values
the difference in error is negligible. With many missing values and correlated columns I lose information by not using KNN or iterative.

</details>